In [1]:
from google.cloud import storage
import pandas as pd
import joblib
import json
from sklearn.metrics import accuracy_score
import os
from feast import FeatureStore

In [6]:
store = FeatureStore(repo_path="feature_repo")

OUTPUT_BUCKET = "mlops-course-week1-unique"

storage_client = storage.Client()

artifact_bucket = storage_client.bucket(OUTPUT_BUCKET)

In [7]:
prefixes = sorted(
    set(
        blob.name.split("/")[0]
        for blob in artifact_bucket.list_blobs()
    )
)

latest_run = prefixes[-1]

In [8]:
tmp_dir = "/tmp/inference"
os.makedirs(tmp_dir, exist_ok=True)

model_file = f"{tmp_dir}/model.joblib"

artifact_bucket.blob(
    f"{latest_run}/model.joblib"
).download_to_filename(model_file)

model = joblib.load(model_file)

In [20]:
online_features = store.get_online_features(

    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],

    entity_rows=[
        {
            "iris_id": 50
        }
    ]

).to_dict()

In [21]:
X = pd.DataFrame({

    "sepal_length":[online_features["sepal_length"][0]],

    "sepal_width":[online_features["sepal_width"][0]],

    "petal_length":[online_features["petal_length"][0]],

    "petal_width":[online_features["petal_width"][0]]

})

preds = model.predict(X)

print(X)
print(preds)

   sepal_length  sepal_width  petal_length  petal_width
0           7.0          3.2           4.7          1.4
['versicolor']


## Verify prediction

In [23]:
raw = pd.read_parquet("feature_repo/data/iris.parquet")

print(raw.loc[50])

sepal_length                              7.0
sepal_width                               3.2
petal_length                              4.7
petal_width                               1.4
species                            versicolor
iris_id                                    50
event_timestamp    2026-07-05 07:34:18.674435
Name: 50, dtype: object
